In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
#import datasets
sessions = pd.read_csv('website_sessions.csv')
pageviews = pd.read_csv('website_pageviews.csv')
orders = pd.read_csv('orders.csv')
order_items = pd.read_csv('order_items.csv')
refunds = pd.read_csv('order_item_refunds.csv')
products = pd.read_csv('products.csv')

In [3]:
#quick overview of the data
#check data quality - shape, missing values, duplications
def audit(df, name):
    print(f"\n{name}")
    print("Shape:", df.shape)
    print(df.info())
    print(df.isnull().sum())
    print(df.duplicated().sum())

audit(sessions,"website_sessions")
audit(pageviews,"website_pageviews")
audit(orders,"orders")


website_sessions
Shape: (472871, 9)
<class 'pandas.DataFrame'>
RangeIndex: 472871 entries, 0 to 472870
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   website_session_id  472871 non-null  int64
 1   created_at          472871 non-null  str  
 2   user_id             472871 non-null  int64
 3   is_repeat_session   472871 non-null  int64
 4   utm_source          389543 non-null  str  
 5   utm_campaign        389543 non-null  str  
 6   utm_content         389543 non-null  str  
 7   device_type         472871 non-null  str  
 8   http_referer        432954 non-null  str  
dtypes: int64(3), str(6)
memory usage: 61.6 MB
None
website_session_id        0
created_at                0
user_id                   0
is_repeat_session         0
utm_source            83328
utm_campaign          83328
utm_content           83328
device_type               0
http_referer          39917
dtype: int64
0

website_pageview

There are approximately 8300 records in website_sessios.csv with missing values in utm_source, utm_campaign & utm_content.

In [4]:
#check unique values in the utm_source, utm_campaign, utm_content columns
source_values = sessions["utm_source"].dropna().unique()
campaign_values = sessions["utm_campaign"].value_counts(dropna=False)
content_values = sessions["utm_content"].value_counts(dropna=False).head(20)

print(f"Unique Source Values: \n {source_values} \n Unique Campaign Values: \n {campaign_values} \n Unique Content Values: \n {content_values}")

Unique Source Values: 
 <ArrowStringArray>
['gsearch', 'bsearch', 'socialbook']
Length: 3, dtype: str 
 Unique Campaign Values: 
 utm_campaign
nonbrand            337615
NaN                  83328
brand                41243
desktop_targeted      5590
pilot                 5095
Name: count, dtype: int64 
 Unique Content Values: 
 utm_content
g_ad_1         282706
NaN             83328
b_ad_1          54909
g_ad_2          33329
b_ad_2           7914
social_ad_2      5590
social_ad_1      5095
Name: count, dtype: int64


The 3 columns with missing values likely track data from campaigns. So the nulls mean So nulls mean:
- direct traffic
- unpaid organic traffic
- untagged referrals

In [5]:
#check traffic sources for null values
sessions.loc[
    sessions["utm_source"].isnull(),
    "http_referer"
].value_counts(dropna=False).head(20)

http_referer
NaN                        39917
https://www.gsearch.com    35202
https://www.bsearch.com     8209
Name: count, dtype: int64

In [6]:
#handle nulls in sessions table
sessions["utm_source"] = sessions["utm_source"].fillna("non_paid")
sessions["utm_campaign"] = sessions["utm_campaign"].fillna("no_campaign")
sessions["utm_content"] = sessions["utm_content"].fillna("no_ad_content")

In [7]:
#examine pageviews for logical inconsistencies in pageview_url column
pageviews.groupby("pageview_url")["created_at"].agg(["min","max","count"])

,min,max,count
pageview_url,,,
/billing,2012-03-19 09:19:52,2013-01-05 20:53:22,3617
/billing-2,2012-09-10 00:13:05,2015-03-19 07:59:07,48441
/cart,2012-03-19 09:14:02,2015-03-19 07:55:03,94953
/home,2012-03-19 08:04:16,2015-03-19 07:59:08,137576
/lander-1,2012-06-19 00:35:54,2013-03-10 23:10:57,47574
/lander-2,2013-01-14 00:28:28,2014-12-27 23:50:35,131170
/lander-3,2013-07-09 00:51:33,2015-03-19 07:55:40,79000
/lander-4,2014-02-02 00:29:20,2014-04-19 23:47:17,9385
/lander-5,2014-08-02 00:22:29,2015-03-19 07:56:29,68166


There're multiple versions of the landing page (/home, /lander-1, /lander-2, etc.) and billing pages (/billing, /billing-2). However, based on above, the website evolved progressively over time e.g. homepage started as home then evolved to lander-1 in 2012 and eventually lander-5 in 2014. They'll be grouped together when mapping funnel stages.

In [8]:
#qualitative validation
#statistical summary
orders[["items_purchased","price_usd","cogs_usd"]].describe()

,items_purchased,price_usd,cogs_usd
count,32313.000000,32313.000000,32313.000000
mean,1.238666,59.991636,22.355406
std,0.426274,17.808771,6.238621
min,1.000000,29.990000,9.490000
25%,1.000000,49.990000,19.490000
50%,1.000000,49.990000,19.490000
75%,1.000000,59.990000,22.490000
max,2.000000,109.980000,41.980000


- Most customers buy single-item orders
- Revenue is driven by mid-priced products (USD50 – USD60 range)
- Cost is largely stable (mean USD 22.355 and median USD 19.49)

In [9]:
#export cleaned tables
sessions.to_csv("clean_website_sessions.csv", index=False)
pageviews.to_csv("clean_website_pageviews.csv", index=False)
orders.to_csv("clean_orders.csv", index=False)